In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# --- Setup ---
sfreq = 1000  # Hz
duration = 1.0  # seconds
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# --- Step 1: Generate low-frequency signal (4 Hz theta) ---
lf_freq = 4
lf_signal = np.sin(2 * np.pi * lf_freq * times)
lf_phase = np.angle(hilbert(lf_signal))  # Extract low-freq phase φ_L(t)

# --- Step 2: Generate amp_mod(t) from word onsets ---
word_onsets = np.zeros(n_samples)
word_locs = np.random.randint(100, 900, size=5)  # Simulated word onsets
word_onsets[word_locs] = 1

kernel_width = int(0.1 * sfreq)  # 100 ms smoothing
kernel = np.exp(-np.linspace(-1, 1, 2 * kernel_width)**2 / 0.05)
kernel = kernel / kernel.max()
amp_mod = np.convolve(word_onsets, kernel, mode='same')
amp_mod = 0.25 + 0.75 * (amp_mod - amp_mod.min()) / (amp_mod.max() - amp_mod.min())

# --- Step 3: Generate independent HF phase signal ---
# HF phase will be partially locked to LF phase during amp_mod(t)
# Build an instantaneous phase trajectory
hf_freq = 80
hf_phase = 2 * np.pi * hf_freq * times

# Inject PPC by blending with LF phase
# PPC model: φ_H(t) = (1 - amp_mod) * φ_H_ind(t) + amp_mod * φ_L(t) * scale
hf_phase_locked = (1 - amp_mod) * hf_phase + amp_mod * (lf_phase + np.pi/2)  # π/2 offset: not identical phase

# --- Step 4: Create PPC-modulated HF signal ---
hf_signal = np.sin(hf_phase_locked) + 0.3 * np.random.randn(n_samples)

# --- Optional diagnostic: phase difference between LF and HF phases ---
phase_diff = np.angle(np.exp(1j * (hf_phase_locked - lf_phase)))

# --- Plot ---
plt.figure(figsize=(10, 4))
plt.plot(times, hf_signal, color='lightgrey', label='Simulated PPC EEG')
plt.plot(times, amp_mod, label='amp_mod(t)', linestyle='-')
plt.plot(times, np.sin(lf_phase), label='sin(ϕ_L(t))', linestyle='--')
plt.plot(times, np.sin(hf_phase_locked), label='sin(ϕ_H(t))', linestyle=':')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('Data-Inspired PPC Simulation (Weissbart-style)')
plt.legend()
plt.tight_layout()
plt.show()
